## Librerías

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargar y ejecutar un modelo LLM pequeño

## Selección del modelo adecuado
En esta práctica utilizaremos el `modelo Qwen2.5-1.5B-Instruct`, un modelo de lenguaje optimizado para seguir instrucciones. Este modelo ofrece un buen equilibrio entre rendimiento y consumo de memoria, permitiendo su uso en CPU.

##  Cargar el modelo
En este apartado cargaremos el modelo y su tokenizer desde Hugging Face. También seleccionaremos automáticamente la precisión más adecuada según el hardware disponible, usando bfloat16 cuando sea posible para reducir el consumo de memoria.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the best dtype for the current hardware
# bfloat16 is preferred (half memory, stable on CPU)
# float32 is the safe fallback
if torch.cuda.is_available():
    dtype = torch.bfloat16
    device_map = "auto"
elif hasattr(torch, "bfloat16"):
    dtype = torch.bfloat16
    device_map = "cpu"
else:
    dtype = torch.float32
    device_map = "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype = dtype,
    device_map = device_map,
)

print(f"Model loaded on {model.device} with dtype {dtype}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

## Verificación del funcionamiento del modelo
En este apartado comprobamos que el modelo funciona correctamente generando una salida simple. También verificamos que no haya errores numéricos (NaN) en los valores internos del modelo.

In [ ]:
# Quick sanity check
test_input = tokenizer("The capital of France is", return_tensors = "pt").to(model.device)

with torch.no_grad():
    logits = model(**test_input).logits
    last_logits = logits[0, -1, :]
    print(f"Logits — min: {last_logits.min().item():.2f}, max: {last_logits.max().item():.2f}")
    print(f"Contains NaN: {torch.isnan(last_logits).any().item()}")
    
    # Generate a few tokens
    out = model.generate(**test_input, max_new_tokens = 10, do_sample = False,
                         pad_token_id = tokenizer.eos_token_id)
    print(f"Output: {tokenizer.decode(out[0], skip_special_tokens = True)}")

## Comprensión de la tokenización
Antes de procesar texto, el modelo lo convierte en tokens, que son representaciones numéricas. El tokenizer se encarga de transformar el texto en estos tokens y de aplicar el formato necesario para el modelo.

# Basic tokenization
text = "Hello, how are you doing today?"
tokens = tokenizer(text, return_tensors = "pt")

print(f"Input text: {text}")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Number of tokens: {tokens['input_ids'].shape[1]}")
print(f"Decoded tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")

## Generación de texto
En este apartado utilizamos el método `generate()` para producir texto a partir de una entrada. También introducimos parámetros que controlan el comportamiento del modelo, como la aleatoriedad y la diversidad de las respuestas.

def generate_response(model, tokenizer, prompt, max_new_tokens = 256):
    """
    Generate a response from the model given a raw text prompt.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        prompt: Input text string
        max_new_tokens: Maximum number of tokens to generate
    
    Returns:
        str: The generated response
    """
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = 0.7,
            top_p = 0.9,
            top_k = 50,
            pad_token_id = tokenizer.eos_token_id,
        )
    
    # Decode only the new tokens (exclude the prompt)
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens = True
    )
    return response


# Test
response = generate_response(model, tokenizer, "The capital of France is")
print(response)

## Uso de plantillas de chat
Los modelos ajustados para instrucciones utilizan un formato de conversación con roles (system, user, assistant). El tokenizer aplica automáticamente este formato mediante plantillas de chat para que el modelo entienda mejor la entrada.

In [ ]:
def chat(model, tokenizer, user_message, system_message = "You are a helpful assistant.",
         max_new_tokens = 256, temperature = 0.7, top_p = 0.9, top_k = 50):
    """
    Send a message using the model's chat template.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        user_message: The user's message
        system_message: The system prompt
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (use 0.0 for greedy)
        top_p: Nucleus sampling threshold
        top_k: Top-k sampling (limits vocabulary at each step)
    
    Returns:
        str: The assistant's response
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, tokenize = False, add_generation_prompt = True
    )
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    
    gen_kwargs = dict(
        max_new_tokens = max_new_tokens,
        pad_token_id = tokenizer.eos_token_id,
    )
    
    if temperature < 0.01:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p
        gen_kwargs["top_k"] = top_k
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens = True
    )
    return response


# Test
response = chat(model, tokenizer, "Explain what a neural network is in 2 sentences.")
print(response)

## Parámetros de generación
El comportamiento del modelo al generar texto puede controlarse mediante parámetros como la temperatura, top-p y top-k. Estos permiten ajustar el nivel de aleatoriedad y diversidad en las respuestas generadas.

In [ ]:
# Compare different temperature values
prompt = "Write a creative name for a coffee shop."

print("=== Temperature 0.3 (near deterministic) ===")
for i in range(3):
    r = chat(model, tokenizer, prompt, max_new_tokens = 30, temperature = 0.3)
    print(f"  Run {i + 1}: {r.strip()}")

print("\n=== Temperature 0.7 (balanced) ===")
for i in range(3):
    r = chat(model, tokenizer, prompt, max_new_tokens = 30, temperature = 0.7)
    print(f"  Run {i + 1}: {r.strip()}")

print("\n=== Temperature 1.2 (creative) ===")
for i in range(3):
    r = chat(model, tokenizer, prompt, max_new_tokens = 30, temperature = 1.2)
    print(f"  Run {i + 1}: {r.strip()}")